# 2. Azure ML Fine-Tuning and Model Registration

Submit reproducible SLM fine-tuning as an Azure ML command job, track the experiment with MLflow, persist outputs in the workspace datastore, and register an immutable candidate model.

## Experimental design

The notebook is the control plane; `lib/train.py` is the remote execution unit. Training consumes a versioned data asset, masks prompt tokens from the loss, adapts attention and MLP projections with rank-stabilized PEFT-compatible settings, evaluates against the validation split, and writes a merged model to a named job output.

**Mandatory production controls**
- Use a dedicated GPU compute cluster with managed identity and minimum nodes set to zero.
- Pin the curated environment and data/model asset versions.
- Keep the test split sealed until Notebook 3. Registration here creates a candidate, not an automatic production promotion.

In [2]:
from datetime import datetime, timezone
from pathlib import Path
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "lib").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "lib").exists():
    raise RuntimeError("Start this notebook from the repository or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from lib.azureml_ops import (
    create_training_environment,
    register_job_model,
    submit_finetuning_job,
)

from lib.config import AzureMLConfig

## Run configuration

Replace immutable asset versions deliberately. The Key Vault fields are names, not secret values. Leave both as `None` only when the base model is public or already cached.

In [3]:
DATA_ASSET_NAME = "raft-instance-security"
DATA_ASSET_VERSION = "20260804.065418"
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct" # 2 GB VRAM
COMPUTE_NAME = "gpu-cluster1"
MANAGED_IDENTITY_CLIENT_ID = "39793817-16f4-45f9-8181-b8e198144c49"
TRAIN_ENVIRONMENT_NAME = "raft-slm-finetuning"
TRAIN_ENVIRONMENT_VERSION = "2"
EXPERIMENT_NAME = "raft-llama32-finetuning"
REGISTERED_MODEL_NAME = "raft-llama32-1b"
REGISTERED_MODEL_VERSION = datetime.now(timezone.utc).strftime("%Y%m%d.%H%M%S")

## Connect and resolve immutable inputs

`DefaultAzureCredential` uses your Azure CLI identity locally and managed identity in Azure ML. No subscription IDs or credentials are stored in notebook output.

In [4]:
config = AzureMLConfig.from_env()
ml_client = config.create_ml_client()
data_asset = ml_client.data.get(DATA_ASSET_NAME, version=DATA_ASSET_VERSION)
print("Data input:", data_asset.id)
print("Compute:", ml_client.compute.get(COMPUTE_NAME).name)

Data input: /subscriptions/ff9fa810-9dbb-4085-9c75-10b2f491bace/resourceGroups/sriks-mlhub-mcaps/providers/Microsoft.MachineLearningServices/workspaces/sriks-aml-ws/data/raft-instance-security/versions/20260804.065418
Compute: gpu-cluster1


## Register the pinned training environment

The environment definition lives in `environments/train-conda.yml`. Increment its version whenever a dependency changes; never mutate an environment used by a completed experiment.

In [5]:
training_environment = create_training_environment(
    ml_client, TRAIN_ENVIRONMENT_NAME, TRAIN_ENVIRONMENT_VERSION
)
print("Environment:", training_environment.id)

Environment: /subscriptions/ff9fa810-9dbb-4085-9c75-10b2f491bace/resourceGroups/sriks-mlhub-mcaps/providers/Microsoft.MachineLearningServices/workspaces/sriks-aml-ws/environments/raft-slm-finetuning/versions/2


>> The above command submits a job for creating the environment. Ensure environment is successfully created before proceeding.

## Submit and observe the remote job

Azure ML snapshots source code and mounts the output on workspace storage. Transformers metrics, parameters, system metrics, validation loss, and perplexity are sent to the workspace MLflow tracking server.

In [5]:
import os
print(os.environ.get("AZURE_KEY_VAULT_NAME"))
print(os.environ.get("AZURE_KEY_VAULT_HF_TOKEN_SECRET_NAME"))

sriksamlkeyvault6b382e43
hftoken


In [ ]:
submitted_job = submit_finetuning_job(
    ml_client=ml_client,
    data_asset=f"azureml:{DATA_ASSET_NAME}:{DATA_ASSET_VERSION}",
    environment=training_environment.id,
    compute=COMPUTE_NAME,
    experiment_name=EXPERIMENT_NAME,
    base_model=BASE_MODEL,
    display_name=f"raft-sft-slm-{REGISTERED_MODEL_VERSION}",
    key_vault_name=os.environ.get("AZURE_KEY_VAULT_NAME"),
    hf_token_secret_name=os.environ.get("AZURE_KEY_VAULT_HF_TOKEN_SECRET_NAME"),
    managed_identity_client_id=MANAGED_IDENTITY_CLIENT_ID,
)

print("Job name:", submitted_job.name)

## Inspect MLflow lineage

Use the experiment view for learning curves and system utilization. Programmatic lookup below makes the association between Azure ML job and MLflow run explicit.

In [6]:
import mlflow

workspace = ml_client.workspaces.get(config.workspace_name)
mlflow.set_tracking_uri(workspace.mlflow_tracking_uri)
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    max_results=10,
    order_by=["start_time DESC"],
)
visible_columns = [
    column
    for column in runs.columns
    if column in {"run_id", "status"}
    or column.startswith("metrics.")
    or column.startswith("params.")
]
runs[visible_columns].head()

,run_id,status,metrics.total_flos,metrics.train_runtime,metrics.eval_steps_per_second,metrics.grad_norm,metrics.train_steps_per_second,metrics.train_samples_per_second,metrics.learning_rate,metrics.loss,...,params.vocab_size,params.adam_epsilon,params.pretraining_tp,params.lr_scheduler_type,params.max_length,params.per_gpu_train_batch_size,params.bos_token_id,params.typical_p,params.max_grad_norm,params.weight_decay
0,joyful_lamp_t97llhznvn,FINISHED,4.368720e+15,3146.5179,0.367,2.14526,0.006,0.207,0.000001,1.1298,...,None,None,None,None,None,None,None,None,None,None
1,ivory_cart_pydrl6bzrd,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,None,None,None,None,None,None,None,None
2,willing_leaf_rpvb67s2p9,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,128256,1e-08,1,linear,20,None,128000,1.0,1.0,0.0
3,sleepy_cat_8kwhc0dbzm,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,None,None,None,None,None,None,None,None
4,frank_spring_5dh0l9gh6p,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,None,None,None,None,None,None,None,None


## Register the candidate model

Registration points directly to the named job output in Azure Storage, retaining job lineage without a local download/re-upload. The candidate remains unpromoted until held-out evaluation and deployment smoke tests pass in Notebook 3.

In [11]:
completed_job = ml_client.jobs.get("joyful_lamp_t97llhznvn") # submitted_job.name)
if completed_job.status != "Completed":
    raise RuntimeError(f"Training did not complete successfully: {completed_job.status}")

registered_model = register_job_model(
    ml_client=ml_client,
    job_name="joyful_lamp_t97llhznvn",
    model_name=REGISTERED_MODEL_NAME,
    version="1",
)

print("Registered candidate:", registered_model.id)

Registered candidate: /subscriptions/ff9fa810-9dbb-4085-9c75-10b2f491bace/resourceGroups/sriks-mlhub-mcaps/providers/Microsoft.MachineLearningServices/workspaces/sriks-aml-ws/models/raft-llama32-1b/versions/1


## Handoff and approval evidence

Retain the data fingerprint, source commit, environment version, job name, MLflow run, model version, training/evaluation curves, and responsible approver. Cost, license acceptance, PII review, red-team results, and rollback ownership belong in the model card before production promotion.

In [ ]:
import os
import subprocess
import pandas as pd
from IPython.display import Markdown, display

source_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    capture_output=True,

    text=True,

    check=True,

).stdout.strip()



data_fingerprint = (data_asset.tags or {}).get("fingerprint", "Unavailable")

job_name = completed_job.name

model_version = str(registered_model.version)

environment_version = str(training_environment.version)



job_runs = runs

if "tags.mlflow.runName" in runs.columns:

    matching_runs = runs[runs["tags.mlflow.runName"].astype(str).str.contains(job_name, na=False)]

    if not matching_runs.empty:

        job_runs = matching_runs

latest_run = job_runs.iloc[0] if not job_runs.empty else None

mlflow_run_id = latest_run["run_id"] if latest_run is not None else "Unavailable"

metric_columns = sorted(column for column in runs.columns if column.startswith("metrics."))

curve_metrics = ", ".join(column.removeprefix("metrics.") for column in metric_columns) or "Unavailable"



lineage_summary = pd.DataFrame(

    {

        "Evidence": [

            data_fingerprint,

            source_commit,

            environment_version,

            job_name,

            mlflow_run_id,

            model_version,

            curve_metrics,

            os.getenv("RESPONSIBLE_APPROVER", "PENDING"),

        ]

    },

    index=[

        "Data fingerprint",

        "Source commit",

        "Environment version",

        "Azure ML job",

        "MLflow run",

        "Model version",

        "Training/evaluation curves",

        "Responsible approver",

    ],

)



model_card_summary = pd.DataFrame(

    {

        "Status / owner": [

            os.getenv("MODEL_CARD_COST_REVIEW", "PENDING"),

            os.getenv("MODEL_CARD_LICENSE_REVIEW", "PENDING"),

            os.getenv("MODEL_CARD_PII_REVIEW", "PENDING"),

            os.getenv("MODEL_CARD_RED_TEAM_RESULTS", "PENDING"),

            os.getenv("MODEL_CARD_ROLLBACK_OWNER", "PENDING"),

        ]

    },

    index=[

        "Cost review",

        "License acceptance",

        "PII review",

        "Red-team results",

        "Rollback ownership",

    ],

)



display(Markdown("## Promotion Evidence Summary"))

display(lineage_summary.style.set_properties(**{"text-align": "left"}))

display(Markdown("### Model Card Readiness"))

display(model_card_summary.style.set_properties(**{"text-align": "left"}))



pending = model_card_summary["Status / owner"].eq("PENDING").sum()

if pending:

    display(Markdown(f"**Promotion status: BLOCKED** - {pending} model-card item(s) remain pending."))

else:

    display(Markdown("**Promotion status: READY FOR APPROVER REVIEW**"))